<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/master/FASE_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FASE 2: Identificación del organismo mediante BLASTn
# ============================================================
# Estrategia: BLASTn contra nt (nucleótidos) de NCBI.
# Buscamos el hit con mayor identidad + cobertura y miramos
# la taxonomía del organismo.

!pip install -q biopython

from Bio.Blast import NCBIWWW, NCBIXML
from Bio import SeqIO
from google.colab import files
import time

# --- 1. Cargar la secuencia (mismo archivo de la Fase 1) ---
uploaded = files.upload()
fasta_file = list(uploaded.keys())[0]
record = SeqIO.read(fasta_file, "fasta")

# --- 2. Ejecutar BLASTn (puede tardar 1-3 minutos) ---
print("Enviando BLASTn a NCBI... (espera)")
result_handle = NCBIWWW.qblast(
    program="blastn",
    database="nt",
    sequence=str(record.seq),
    hitlist_size=10,          # top 10
    expect=1e-5,
    format_type="XML"
)

# Guardar resultado para no repetir
with open("blastn_result.xml", "w") as f:
    f.write(result_handle.read())
result_handle.close()

# --- 3. Parsear resultados ---
with open("blastn_result.xml") as f:
    blast_record = NCBIXML.read(f)

print(f"\n{'='*60}")
print(f"Query: {blast_record.query}")
print(f"{'='*60}\n")

for i, alignment in enumerate(blast_record.alignments[:5], 1):
    hsp = alignment.hsps[0]
    identity = (hsp.identities / hsp.align_length) * 100
    coverage = (hsp.align_length / blast_record.query_length) * 100
    print(f"Hit {i}: {alignment.title[:100]}...")
    print(f"   Longitud hit: {alignment.length}")
    print(f"   Identidad: {identity:.1f}% | Cobertura: {coverage:.1f}%")
    print(f"   E-value: {hsp.expect:.2e}")
    print(f"   Score: {hsp.score}")
    print("-" * 50)

print("\n✅ Interpreta el mejor hit (mayor identidad + cobertura).")
print("   El nombre del organismo suele estar en el título del alignment.")
print("   Ejemplo: 'Escherichia coli strain ...', 'Salmonella enterica ...', etc.")